### Import ###

In [1]:
import pandas as pd
import os
import math
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import numpy as np

### Compressing Study Data by any Seconds
#### The smartest option is 0.1, since it then can be merged to make any other portion, 0.2, ... 0.9 sec.

In [2]:
def split_into_chunks(df, groupby_column, num_chunks):
    # Group the data
    grouped_data = df.groupby(groupby_column)
    
    # Function to split each group into equal-sized chunks
    def split_group(group):
        chunk_size = len(group) // num_chunks
        labels = [i for i in range(1, num_chunks + 1) for _ in range(chunk_size)]
        remainder = len(group) % num_chunks
        labels += [num_chunks] * remainder
        group['quantile'] = labels[:len(group)]
        return group
    # Apply the function to each group and concatenate the results
    result = grouped_data.apply(split_group)
    
    return result.reset_index(drop=True)
    
# Ask the user for the time duration to turn it into the number of chunks
x = float(input("Enter the segment size in seconds (e.g., 0.2): "))
quantile = int(1/x)
# type(quantile)

Enter the segment size in seconds (e.g., 0.2):  0.1


In [3]:
data_by_sec = pd.DataFrame()

# Get the current working directory
current_directory = os.getcwd()

# Specify the relative folder path
folder_path = os.path.join(current_directory, 'textfiles')

In [63]:
#MAB    
# Loop over all files in the folder
# Specify the specific file name you want to read
file_name = 'PID_046_Low_2024-8-26_16_55_4'

# Construct the full path to the specific file
file_path = os.path.join(folder_path, file_name)
df = pd.read_csv(file_path, header=None, names=['line'])

# Split the filename
print(file_name)
data_line = file_name.split('_')
print(data_line[1:7])

# Find the start and end index for each AGV
prefix_range = range(1, 17)
prefix_list = list(f"{'AGVname=AGV'}{x}" for x in prefix_range)
start_ind = []
for prefix in prefix_list:
    matching_indices = df[df['line'].str.startswith(prefix)].index
    if not matching_indices.empty:
        start_ind.append(matching_indices[0])
    else:
        # Handle the case where no match is found
        print(f"No match found for prefix: {prefix}")
        start_ind.append(None)  # or handle it in another appropriate way

end_ind = df[df['line'].str.startswith('CheckPoint/Route')].index

user_data = pd.DataFrame(columns=['User_X', 'User_Y', 'User_Z', 'Timestamp', 'User_Pitch', 'User_Yaw', 'User_Roll', 'U_X', 'U_Y', 'U_Z','quantile'])
gaze_data = pd.DataFrame(columns=['Target', 'Timestamp', 'GazeOrigin_X', 'GazeOrigin_Y', 'GazeOrigin_Z','GazeDirection_X','GazeDirection_Y','GazeDirection_Z','Confidence','quantile'])
AGV_data = pd.DataFrame(columns=['AGVname', 'AGV_X', 'AGV_Y', 'AGV_Z', 'AGV_Pitch', 'AGV_Yaw', 'AGV_Roll', 'AGV_spd', 'Timestamp','quantile'])
    
for i in range(16):
    print(f"Processing index: {i}")
    print(f"start_ind[{i}] = {start_ind[i]}")
    print(f"end_ind[{i}] = {end_ind[i]}")
    
    # Check if the indices are not None
    if start_ind[i] is not None and end_ind[i] is not None:
        for j in range(start_ind[i], end_ind[i]):
            line = df['line'].iloc[j]

            if line[0] == 'U':
                fields = line.split()
                d = []
                for field in fields:
                    parts = field.split('=')
                    key, value = parts
                    d.append(value)
                user_data.loc[len(user_data)] = d
                #print(d)
            elif line[0] == 'E':
                fields = line.split()
                d = []
                for field in fields:
                    parts = field.split('=')
                    key, value = parts
                    d.append(value)
                gaze_data.loc[len(gaze_data)] = d
                #print(d)
            elif line[0] == 'A':
                fields = line.split()
                d = []
                for field in fields:
                    parts = field.split('=')
                    key, value = parts
                    d.append(value)
                AGV_data.loc[len(AGV_data)] = d
                #print(d)
    else:
        # if an AGV data is not recorded, this message will appear and processing the data for that will be skipped
        print(f"Skipping index {i} due to None value in start_ind or end_ind")
            #print(d)
    print('number of quantiles based on the sec entry:', quantile)
    user_data = user_data.astype({'User_X':float, 'User_Y':float, 'User_Z':float, 'User_Pitch':float, 'User_Yaw':float, 'User_Roll':float, 'U_X':float, 'U_Y':float, 'U_Z':float})
    user_data['Timestamp'] = pd.to_datetime(user_data['Timestamp'], format='%H:%M:%S').dt.time
    user_data['quantile'] = quantile
    # Call the function to split the data into the specified number of chunks based on Timestamp, and add the chunk lable as an indicator
    user_data = split_into_chunks(user_data, 'Timestamp', quantile)
    # Calculate the mean of the user_data columns for each unique pair of quntile and timestamp
    grouped_user = user_data.groupby(['Timestamp', 'quantile']).mean().reset_index()
    user_data.drop(columns=['quantile'], inplace=True)
    print('user_data done for AGV', i+1)
    
    gaze_data = gaze_data.astype({'GazeOrigin_X':float, 'GazeOrigin_Y':float, 'GazeOrigin_Z':float, 'GazeDirection_X':float, 'GazeDirection_Y':float,'GazeDirection_Z':float, 'Confidence':float})
    gaze_data['Timestamp'] = pd.to_datetime(gaze_data['Timestamp'], format='%H:%M:%S').dt.time
    # Call the function to split the gaze_data into the specified number of chunks based on Timestamp, and add the chunk lable as an indicator
    gaze_data = split_into_chunks(gaze_data, 'Timestamp', quantile)
    # Calculate the mean of the gaze_data columns for each unique pair of quntile and timestamp
    grouped_gaze = gaze_data.groupby(['Timestamp', 'quantile'])[['GazeOrigin_X', 'GazeOrigin_Y', 'GazeOrigin_Z','GazeDirection_X','GazeDirection_Y','GazeDirection_Z','Confidence']].mean().reset_index()
    starts_with_agv = gaze_data.groupby(['Timestamp', 'quantile'])['Target'].apply(lambda x: any(s.startswith('AGV') for s in x.values)).reset_index(name='Gaze_on_AGV')
    grouped_gaze = pd.merge(grouped_gaze, starts_with_agv, on=['Timestamp', 'quantile'])
    gaze_data.drop(columns=['quantile'], inplace=True)
    print('gaze_data done for AGV', i+1)
    
    AGV_data = AGV_data.astype({'AGV_X':float, 'AGV_Y':float, 'AGV_Z':float, 'AGV_Pitch':float, 'AGV_Yaw':float, 'AGV_Roll':float,'AGV_spd':float})
    AGV_data['Timestamp'] = pd.to_datetime(AGV_data['Timestamp'], format='%H:%M:%S').dt.time
    AGV_data = split_into_chunks(AGV_data, 'Timestamp', quantile)
    grouped_agv = AGV_data.groupby(['Timestamp', 'quantile'])[['AGV_X', 'AGV_Y', 'AGV_Z', 'AGV_Pitch', 'AGV_Yaw', 'AGV_Roll', 'AGV_spd']].mean().reset_index()
    majority_df = AGV_data.groupby(['Timestamp', 'quantile'])['AGVname'].apply(lambda x: x.mode().iloc[0]).reset_index(name='AGVname')
    grouped_agv = pd.merge(grouped_agv, majority_df, on=['Timestamp', 'quantile'])
    AGV_data.drop(columns=['quantile'], inplace=True)
    print('AGV_data done for AGV', i+1)
    
    merged_df = grouped_user.merge(grouped_gaze, on=['Timestamp', 'quantile']).merge(grouped_agv, on=['Timestamp', 'quantile'])
    print('Merging three parts of the dataset based on AGV', i+1)
    
    # Creating a DataFrame from the repeated random list
    PID_DRate_df = pd.DataFrame([data_line[1:3]] * len(merged_df), columns=['PID','DRate'])

    # Concatenating the original DataFrame with the additional columns
    final_df = pd.concat([merged_df, PID_DRate_df], axis=1)
    data_by_sec = pd.concat([data_by_sec, final_df], ignore_index=True)
    # Save the merged dataset to a new CSV file with a timestamp in the filename

PID_046_Low_2024-8-26_16_55_4
['046', 'Low', '2024-8-26', '16', '55', '4']
Processing index: 0
start_ind[0] = 24756
end_ind[0] = 23416
number of quantiles based on the sec entry: 10
user_data done for AGV 1
gaze_data done for AGV 1
AGV_data done for AGV 1
Merging three parts of the dataset based on AGV 1
Processing index: 1
start_ind[1] = 3058
end_ind[1] = 35113
number of quantiles based on the sec entry: 10
user_data done for AGV 2
gaze_data done for AGV 2
AGV_data done for AGV 2
Merging three parts of the dataset based on AGV 2
Processing index: 2
start_ind[2] = 35739
end_ind[2] = 48063
number of quantiles based on the sec entry: 10
user_data done for AGV 3
gaze_data done for AGV 3
AGV_data done for AGV 3
Merging three parts of the dataset based on AGV 3
Processing index: 3
start_ind[3] = 49166
end_ind[3] = 60387
number of quantiles based on the sec entry: 10
user_data done for AGV 4
gaze_data done for AGV 4
AGV_data done for AGV 4
Merging three parts of the dataset based on AGV 4
Pr

In [64]:
# Concatenating the new preprocessed DataFrame with ISU_data_by_0.1_sec by adding it to the end of the dataset
# Reading the main CSV file
main_data = pd.read_csv('ISU_data_by_0.1_sec.csv')

# Merging the new processed data with the main CSV file
merged_data = pd.concat([main_data, data_by_sec], axis=0)

# Print the merged data
print('After merging:')
print(merged_data)

# Generate a timestamp
# timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# Save the merged dataset to a new CSV file with a timestamp in the filename
merged_data.to_csv(f'ISU_data_by_{x}_sec.csv', index=False)

After merging:
           User_X    User_Y   User_Z  User_Pitch   User_Yaw  User_Roll  \
0        5051.235  8610.830 -209.725  -21.416662 -92.196247  -0.159610   
1        5051.235  8610.830 -209.725  -21.168361 -92.384752   0.290698   
2        5051.235  8610.830 -209.725  -21.240655 -92.761660   0.339485   
3        5051.235  8610.830 -209.725  -21.004733 -93.120707   0.251285   
4        5051.235  8610.830 -209.725  -20.717933 -91.225685   0.512156   
...           ...       ...      ...         ...        ...        ...   
2652450  4814.045  8544.847 -209.725    9.361614 -83.257447   2.434928   
2652451  4814.045  8544.847 -209.725    8.837808 -85.161383   1.814595   
2652452  4814.045  8544.847 -209.725    8.142000 -88.676587   0.913597   
2652453  4814.045  8544.847 -209.725    7.388442 -92.010121   0.207441   
2652454  4814.045  8544.847 -209.725    6.817216 -95.838622  -0.021608   

               U_X         U_Y         U_Z  GazeOrigin_X  ...       AGV_Z  \
0        60.524000 

### Compressing Study Data by One Second

In [56]:
#Original 

# Loop over all files in the folder
for filename in os.listdir(folder_path):
    file_path = os.path.join(folder_path, filename)
    df = pd.read_csv(file_path, header=None, names=['line'])

    # Split the filenames
    print(filename)
    data_line = filename.split('_')
    print(data_line[1:3])

    # Find the start and end index for each AGV
    prefix_range = range(1, 17)
    prefix_list = list(f"{'AGVname=AGV'}{x}" for x in prefix_range)
    start_ind = []
    for prefix in prefix_list:
        start_ind.append(df[df['line'].str.startswith(prefix)].index[0])
    end_ind = df[df['line'].str.startswith('CheckPoint/Route')].index

    user_data = pd.DataFrame(columns=['User_X', 'User_Y', 'User_Z', 'Timestamp', 'User_Pitch', 'User_Yaw', 'User_Roll', 'U_X', 'U_Y', 'U_Z'])
    gaze_data = pd.DataFrame(columns=['Target', 'Timestamp', 'GazeOrigin_X', 'GazeOrigin_Y', 'GazeOrigin_Z','GazeDirection_X','GazeDirection_Y','GazeDirection_Z','Confidence'])
    AGV_data = pd.DataFrame(columns=['AGVname', 'AGV_X', 'AGV_Y', 'AGV_Z', 'AGV_Pitch', 'AGV_Yaw', 'AGV_Roll', 'AGV_spd', 'Timestamp'])

    for i in range(16):
        print(i)
        for j in range(start_ind[i], end_ind[i]):
            line = df['line'].iloc[j]

            if line[0]=='U':
                fields = line.split()
                d=[]
                for field in fields:
                    # Split the field into a key and value using '='
                    parts = field.split('=')
                    key, value = parts
                    d.append(value)
                user_data.loc[len(user_data)] = d
            elif line[0]=='E':
                fields = line.split()
                d=[]
                for field in fields:
                    parts = field.split('=')
                    key, value = parts
                    d.append(value)
                gaze_data.loc[len(gaze_data)] = d
            elif line[0]=='A':
                fields = line.split()
                d=[]
                for field in fields:
                    parts = field.split('=')
                    key, value = parts
                    d.append(value)
                AGV_data.loc[len(AGV_data)] = d
    
    user_data = user_data.astype({'User_X':float, 'User_Y':float, 'User_Z':float, 'User_Pitch':float, 'User_Yaw':float, 'User_Roll':float, 'U_X':float, 'U_Y':float, 'U_Z':float})
    user_data['Timestamp'] = pd.to_datetime(user_data['Timestamp'], format='%H:%M:%S').dt.time
    grouped_user = user_data.groupby('Timestamp').mean().reset_index()
    
    gaze_data = gaze_data.astype({'GazeOrigin_X':float, 'GazeOrigin_Y':float, 'GazeOrigin_Z':float, 'GazeDirection_X':float, 'GazeDirection_Y':float,'GazeDirection_Z':float, 'Confidence':float})
    gaze_data['Timestamp'] = pd.to_datetime(gaze_data['Timestamp'], format='%H:%M:%S').dt.time
    grouped_gaze = gaze_data.groupby('Timestamp')[['GazeOrigin_X', 'GazeOrigin_Y', 'GazeOrigin_Z','GazeDirection_X','GazeDirection_Y','GazeDirection_Z','Confidence']].mean().reset_index()
    starts_with_agv = gaze_data.groupby('Timestamp')['Target'].apply(lambda x: any(s.startswith('AGV') for s in x.values)).reset_index(name='Gaze_on_AGV')
    grouped_gaze = pd.merge(grouped_gaze, starts_with_agv, on='Timestamp')
    print('All the way to the gaze_data')

    AGV_data = AGV_data.astype({'AGV_X':float, 'AGV_Y':float, 'AGV_Z':float, 'AGV_Pitch':float, 'AGV_Yaw':float, 'AGV_Roll':float,'AGV_spd':float})
    AGV_data['Timestamp'] = pd.to_datetime(AGV_data['Timestamp'], format='%H:%M:%S').dt.time
    grouped_agv = AGV_data.groupby('Timestamp')[['AGV_X', 'AGV_Y', 'AGV_Z', 'AGV_Pitch', 'AGV_Yaw', 'AGV_Roll', 'AGV_spd']].mean().reset_index()
    majority_df = AGV_data.groupby('Timestamp')['AGVname'].apply(lambda x: x.mode().iloc[0]).reset_index(name='AGVname')
    grouped_agv = pd.merge(grouped_agv, majority_df, on='Timestamp')

    merged_df = grouped_user.merge(grouped_gaze, on='Timestamp').merge(grouped_agv, on='Timestamp')

    # Creating a DataFrame from the repeated random list
    PID_SCN_df = pd.DataFrame([data_line[1:3]] * len(merged_df), columns=['PID','SCN'])


    # Concatenating the original DataFrame with the additional columns
    final_df = pd.concat([merged_df, PID_SCN_df], axis=1)
    data_by_sec = pd.concat([data_by_sec, final_df], ignore_index=True)

PID_001_High_2024-5-3_14_28_53
['001', 'High']
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14


KeyboardInterrupt: 

In [ ]:
data_by_sec.to_csv(f'ISU_data_by_sec.csv', index=False)

### Compressing the 0.1 Seconds Dataset to 0.3 Seconds 
#### 0.3 seconds is an exmaple, you can pick any other portion and follow the below pattern.

In [2]:
study_data = pd.read_csv('ISU_data_by_0.1_sec.csv')

In [3]:
def average_quantiles(group, suffix):
    numeric_columns = group.select_dtypes(include = np.number).columns
    averaged = group[numeric_columns].mean() # Averaging only numeric columns
    original_timestamp = group['Timestamp'].iloc[0]
    new_timestamp = f"{original_timestamp.strftime('%H:%M:%S')}{suffix}"
    averaged['Timestamp'] = new_timestamp
    averaged['quantile'] = ', '.join(map(str, sorted(group['quantile'].unique())))
    averaged['AGVname'] = group['AGVname'].iloc[0]
    averaged['DRate'] = group['DRate'].iloc[0]
    averaged['PID'] = group['PID'].iloc[0]
    # print(quantile_group, new_timestamp, suffix)
    return pd.DataFrame([averaged])

# Converting the Timestamp column to datetime format
study_data['Timestamp'] = pd.to_datetime(study_data['Timestamp'], format = '%H:%M:%S')

# Group and process for quantiles 1, 2, 3
grouped_1 = study_data[study_data['quantile'].isin([1, 2, 3])]
result_1 = grouped_1.groupby(['AGVname','PID','DRate','Timestamp']).apply(lambda x:average_quantiles(x, ".0")).reset_index(drop=True)

# Group and process for quantiles 1, 2, 3
grouped_2 = study_data[study_data['quantile'].isin([4, 5, 6])]
result_2 = grouped_2.groupby(['AGVname','PID','DRate','Timestamp']).apply(lambda x:average_quantiles(x, ".33")).reset_index(drop=True)

# Group and process for quantiles 1, 2, 3
grouped_3 = study_data[study_data['quantile'].isin([7, 8, 9, 10])]
result_3 = grouped_3.groupby(['AGVname','PID','DRate','Timestamp']).apply(lambda x:average_quantiles(x, ".67")).reset_index(drop=True)

# Combine the results into a new dataset
study_data_3 = pd.concat([result_1, result_2, result_3]).reset_index(drop = True)

In [4]:
# Get the list of columns in the dataset
columns_list = study_data_3.columns.tolist()

# Print the list of columns
print(columns_list)

['User_X', 'User_Y', 'User_Z', 'User_Pitch', 'User_Yaw', 'User_Roll', 'U_X', 'U_Y', 'U_Z', 'GazeOrigin_X', 'GazeOrigin_Y', 'GazeOrigin_Z', 'GazeDirection_X', 'GazeDirection_Y', 'GazeDirection_Z', 'Confidence', 'AGV_X', 'AGV_Y', 'AGV_Z', 'AGV_Pitch', 'AGV_Yaw', 'AGV_Roll', 'AGV_spd', 'quantile', 'PID', 'Timestamp', 'AGVname', 'DRate']


In [5]:
# Sort the dataset
study_data_3 = study_data_3.sort_values(
    by = ['PID', 'DRate', 'AGVname', 'Timestamp'],
    ascending = [True, True, True, True]
)

In [6]:
study_data_3.shape

(85446, 28)

In [7]:
study_data_3.iloc[40000:40050]

,User_X,User_Y,User_Z,User_Pitch,User_Yaw,User_Roll,U_X,U_Y,U_Z,GazeOrigin_X,...,AGV_Z,AGV_Pitch,AGV_Yaw,AGV_Roll,AGV_spd,quantile,PID,Timestamp,AGVname,DRate
64881,5054.758365,6413.654218,-209.725,-12.148172,71.011541,5.533205,78.141448,44.803440,229.368083,4976.876238,...,-283.142484,3.887649e-01,0.047525,0.000440,1.504346e+01,"7, 8, 9, 10",9,14:42:03.67,AGV13,Low
8041,5039.949524,6441.690286,-209.725,-13.368427,73.129009,5.909165,75.315714,45.074238,229.352000,4965.258476,...,-283.142381,3.824130e-01,0.047454,0.000456,1.502074e+01,"1, 2, 3",9,14:42:04.0,AGV13,Low
36404,5028.802667,6466.180619,-209.725,-8.104115,77.558400,6.040222,63.125143,44.741762,231.299190,4965.975619,...,-283.143333,3.770764e-01,0.047578,0.000628,1.500224e+01,"4, 5, 6",9,14:42:04.33,AGV13,Low
64882,5027.576000,6468.777000,-209.725,-6.275094,87.556400,7.747005,54.276393,43.080671,231.347071,4973.299460,...,-283.141504,3.936780e-01,0.047909,0.000643,1.505683e+01,"7, 8, 9, 10",9,14:42:04.67,AGV13,Low
8042,5027.576000,6468.777000,-209.725,-4.210432,94.983735,6.615803,60.269524,41.497952,231.922048,4967.306905,...,-283.142524,3.766771e-01,0.048194,0.000607,1.501536e+01,"1, 2, 3",9,14:42:05.0,AGV13,Low
36405,5027.576000,6468.777000,-209.725,-3.610363,100.979067,2.537019,68.319571,40.059619,232.139762,4959.257143,...,-283.141810,3.935184e-01,0.048185,0.000340,1.503852e+01,"4, 5, 6",9,14:42:05.33,AGV13,Low
64883,5027.576000,6468.777000,-209.725,2.471697,102.477097,-3.153423,74.342595,36.482484,233.092187,4953.234409,...,-283.142345,3.812752e-01,0.048181,0.000518,1.503133e+01,"7, 8, 9, 10",9,14:42:05.67,AGV13,Low
8043,5027.576000,6468.777000,-209.725,18.744860,97.406546,1.052188,73.155667,31.237571,235.733762,4954.420714,...,-283.110429,3.910757e-01,0.048465,0.000660,1.502186e+01,"1, 2, 3",9,14:42:06.0,AGV13,Low
36406,5027.576000,6468.777000,-209.725,7.593917,114.967911,11.378335,65.773143,30.670905,234.160810,4961.803381,...,-283.137667,3.867047e-01,0.048718,0.000657,1.505512e+01,"4, 5, 6",9,14:42:06.33,AGV13,Low
64884,5027.576000,6468.777000,-209.725,-7.493248,103.261754,2.990164,68.841607,35.243000,231.502250,4958.721004,...,-283.139107,3.933815e-01,0.049068,0.000592,1.506173e+01,"7, 8, 9, 10",9,14:42:06.67,AGV13,Low


In [8]:
study_data_3.to_csv(f'ISU_data_by_0.3_sec.csv', index=False)